#Análise do uso de dalimumabe na Psoríase em SC (2020 a 2025) - SUS
Base de dados: SIA_SUS - dados de APAC

Análise da dispensações de medicamentos de alto custo para **Psoríase (`CID L40`)** ao longo de **6 anos completos (72 meses: Janeiro/2020 a Dezembro/2025)** 
Análise realizada de forma longitudinal acompanhando o paciente individualmente.

Na mesa de trabalho do auditor existem:

* **6 Caixas Anuais Separadas (`apac_2020` até `apac_2025`):** Cada caixa guarda os 12 meses completos daquele ano (`8.238` a `24.766` fichas por ano).
* **1 Arquivo Geral Unificado (`apac_psoriase`):** Contém todas as **94.370 fichas** dos 72 meses já empilhadas em um único lugar para rastrear cada paciente do início ao fim.

### As variáveis da nossa história

| Coluna no SUS | O que ela significa na auditoria |
| :--- | :--- |
| **`AP_CNSPCN`** | **O Paciente:** Número único de 15 dígitos do Cartão SUS (`700...`) |
| **`AP_CMP`** | **O Mês/Ano (Competência):** Formato `AAAAMM` (ex: `202001` = Jan/2020 até `202512` = Dez/2025) |
| **`AP_PRIPAL`** | **O Medicamento (SIGTAP):** Os 4 códigos do **Adalimumabe 40 mg** são `'0604380011'`, `'0604380062'`, `'0604380127'` e `'0604380135'` |
| **`AP_CIDPRI`** | **A Doença (CID-10):** `L400` (Psoríase vulgar), `L401` (pustulosa), `L404` (gutata), `L408` (outras) |
| **`AP_AUTORIZ`** | **O Número da Ficha APAC:** Protocolo de autorização daquela entrega |

---

### Etapa 0: Colocando as caixas de 2020 a 2025 na memória RAM

O código abaixo usa a função `ativar_sql()` do arquivo `motor_datasus.py` para carregar os 6 anos completos (`apac_2020` a `apac_2025`) e criar a tabela consolidada **`apac_psoriase`** na memória RAM.

In [4]:
# Carrega os 6 anos completos de Psoríase (2020 a 2025) na memória RAM e ativa o %%sql
import importlib, motor_datasus
importlib.reload(motor_datasus)
motor_datasus.ativar_sql()

Carregado na RAM: apac_sc_2020.csv -> tabela SQL 'apac_2020' (8238 linhas)
Carregado na RAM: apac_sc_2021.csv -> tabela SQL 'apac_2021' (11941 linhas)
Carregado na RAM: apac_sc_2022.csv -> tabela SQL 'apac_2022' (13873 linhas)
Carregado na RAM: apac_sc_2023.csv -> tabela SQL 'apac_2023' (16016 linhas)
Carregado na RAM: apac_sc_2024.csv -> tabela SQL 'apac_2024' (19536 linhas)
Carregado na RAM: apac_sc_2025.csv -> tabela SQL 'apac_2025' (24766 linhas)
Tabela consolidada criada na RAM: 'apac_psoriase' (94370 linhas somando todos os anos)
Base higienizada carregada na RAM: 'apac_sc_psoriase_limpa.csv' -> tabela SQL 'apac_psoriase_limpa' (94145 linhas)
Dicionário carregado na RAM: 'dicionarios\sigtap_psoriase.csv' -> tabela SQL 'dicionario_sigtap' (22 linhas)
Dicionário carregado na RAM: 'dicionarios\ibge_municipios_sc.csv' -> tabela SQL 'dicionario_ibge' (295 linhas)


---

### 1. `INNER JOIN` (Pacientes que estavam em tratamento em 2024 e continuaram em 2025)

O auditor quer comparar duas caixas anuais usando **apenas uma variável de ligação**: o paciente (`AP_CNSPCN`).
> *"Quais pacientes com Psoríase tinham ficha na caixa de **2024** e continuaram retirando medicamento na caixa de **2025**?"*

* **`FROM apac_2024 AS a24 INNER JOIN apac_2025 AS a25 ON a24.AP_CNSPCN = a25.AP_CNSPCN`**: O auditor procura o mesmo número de paciente nas duas caixas e grampeia os registros.
* Usamos **`COUNT(DISTINCT a24.AP_CNSPCN)`** para **contar** quantos pacientes únicos permaneceram ativos nos dois anos (sem repetir o mesmo CNS quando ele tem várias fichas no ano e sem precisar listar os números de cartão).

In [ ]:
%%sql
-- 1. INNER JOIN: Quantidade de pacientes únicos presentes em 2024 E que continuaram em 2025

SELECT
    COUNT(DISTINCT a24.AP_CNSPCN) AS qtd_pacientes_2024_e_2025
FROM apac_2024 AS a24
INNER JOIN apac_2025 AS a25
    ON a24.AP_CNSPCN = a25.AP_CNSPCN;

,qtd_pacientes_2024_e_2025
0,2122


In [ ]:
%%sql
-- 1. INNER JOIN: Quantidade de pacientes únicos presentes em 2023 E que continuaram em 2024

SELECT
    COUNT(DISTINCT a23.AP_CNSPCN) AS qtd_pacientes_2023_e_2024
FROM apac_2023 AS a23
INNER JOIN apac_2024 AS a24
    ON a23.AP_CNSPCN = a24.AP_CNSPCN

,qtd_pacientes_2023_e_2024
0,1664


---

### 2. `LEFT JOIN` (Auditoria de Abandono entre Anos: estava em 2024, mas sumiu em 2025)

Agora o auditor faz a pergunta inversa — a base do cálculo de **abandono entre um ano e outro**:
> *"Quais pacientes retiraram medicamento em **2024**, mas **não aparecem em nenhum mês de 2025**?"*

1. **`FROM apac_2024 AS a24 LEFT JOIN apac_2025 AS a25 ON a24.AP_CNSPCN = a25.AP_CNSPCN`**: O auditor protege todos os pacientes de 2024 (esquerda) e procura cada um na caixa de 2025 (direita).
2. Se o paciente parou de ir à farmácia e nunca apareceu em 2025, o lado direito fica vazio (**`NULL`**).
3. **`WHERE a25.AP_CNSPCN IS NULL`**: Na triagem, o auditor separa exatamente os pacientes que desapareceram em 2025.

In [39]:
%%sql
-- 2. LEFT JOIN: Pacientes que tinham ficha em 2024 e ficaram sem nenhuma ficha em 2025 (NULL)
SELECT DISTINCT
    a24.AP_CNSPCN AS cns_paciente_que_saiu_em_2025
FROM apac_2024 AS a24
LEFT JOIN apac_2025 AS a25
    ON a24.AP_CNSPCN = a25.AP_CNSPCN
WHERE a25.AP_CNSPCN IS NULL
LIMIT 10;

,cns_paciente_que_saiu_em_2025
0,700778721977131
1,700257465752956
2,700425566559969
3,700632997765831
4,700654227218381
5,700570161741590
6,700529506115025
7,700211507180950
8,700058838553811
9,700611912565884


---

### 3. `WHERE` (A triagem na portaria: separando apenas quem usa Adalimumabe)

Como a nossa tabela `apac_psoriase` contém todas as 94.370 fichas de Psoríase (`L40`) de 2020 a 2025, o auditor agora avisa o porteiro (`WHERE`):
> *"Deixe passar apenas as fichas em que o medicamento (`AP_PRIPAL`) seja **Adalimumabe 40 mg**."*

* O **`WHERE`** lê ficha por ficha antes de qualquer agrupamento e filtra os 4 códigos do Adalimumabe no SUS: `'0604380011'`, `'0604380062'`, `'0604380127'` e `'0604380135'`.

In [40]:
%%sql
-- 3. WHERE: Filtra na portaria apenas as fichas de Adalimumabe para Psoríase (2020 a 2025)
SELECT
    AP_CMP AS ano_mes,
    AP_CNSPCN AS cns_paciente,
    AP_CIDPRI AS cid_psoriase,
    AP_PRIPAL AS codigo_adalimumabe,
    AP_AUTORIZ AS ficha_apac
FROM apac_psoriase
WHERE AP_PRIPAL IN ('0604380011', '0604380062', '0604380127', '0604380135')
LIMIT 10;

,ano_mes,cns_paciente,cid_psoriase,codigo_adalimumabe,ficha_apac
0,202001,700168187668344,L400,0604380062,4219203747737
1,202001,700916136516178,L400,0604380062,4220200077599
2,202001,700815276617998,L400,0604380062,4220200107410
3,202002,700916136516178,L400,0604380062,4220200077599
4,202002,700612970166506,L400,0604380062,4220200468627
5,202002,700527448365900,L400,0604380062,4220200397810
6,202002,700685491733267,L400,0604380062,4220200416982
7,202002,700276571307708,L400,0604380062,4220200513639
8,202002,700027520079336,L400,0604380062,4220200416454
9,202002,700690809810071,L400,0604380062,4220200427696


---

### 4. `GROUP BY` (A Capa da Pasta Completa do Paciente de Adalimumabe: 2020 a 2025)

Agora o auditor pega todas as fichas de **Adalimumabe** (filtradas na portaria pelo `WHERE`), cria **uma pasta suspensa para cada paciente (`GROUP BY AP_CNSPCN`)** e anota na capa exatamente os 6 dados necessários para estudar a linha do tempo:

1. **`AP_CNSPCN AS paciente`**: O número único do Cartão SUS do paciente.
2. **`MAX(AP_CIDPRI) AS cid10`**: O código CID-10 da Psoríase (`L400`, `L401`, `L404` ou `L408`).
3. **`'Adalimumabe 40mg' AS adalimumabe`**: Identificação do medicamento analisado.
4. **`MIN(AP_CMP) AS data_inicio`**: O primeiro mês/ano (`AAAAMM`) em que o paciente buscou o medicamento.
5. **`MAX(AP_CMP) AS data_final`**: O último mês/ano (`AAAAMM`) em que o paciente buscou o medicamento.
6. **`COUNT(DISTINCT AP_CMP) AS total_meses_utilizados`**: Quantos meses diferentes ele retirou o medicamento entre 2020 e 2025.
7. **`COUNT(DISTINCT AP_AUTORIZ) AS qtd_apacs_distintas`**: Quantos números distintos de autorização APAC (`AP_AUTORIZ`) foram emitidos para o paciente (lembrando que uma mesma autorização costuma valer por até 3 meses seguidos).

In [41]:
%%sql
-- 4. GROUP BY: Resumo na capa da pasta com paciente, CID-10, Adalimumabe, data de início, data final e total de meses
SELECT
    AP_CNSPCN AS paciente,
    MAX(AP_CIDPRI) AS cid10,
    'Adalimumabe 40mg' AS medicamento,
    MIN(AP_CMP) AS data_inicio,
    MAX(AP_CMP) AS data_final,
    COUNT(DISTINCT AP_CMP) AS total_meses_utilizados,
    COUNT(DISTINCT AP_AUTORIZ) AS qtd_apacs_distintas
FROM apac_psoriase
WHERE AP_PRIPAL IN ('0604380011', '0604380062', '0604380127', '0604380135')
GROUP BY AP_CNSPCN
ORDER BY total_meses_utilizados DESC
LIMIT 15;

,paciente,cid10,medicamento,data_inicio,data_final,total_meses_utilizados,n_apacs
0,700815276617998,L400,Adalimumabe 40mg,202001,202512,72,72
1,700939070658737,L400,Adalimumabe 40mg,202002,202512,71,71
2,700729128126364,L400,Adalimumabe 40mg,202002,202512,70,70
3,700653121751680,L400,Adalimumabe 40mg,202003,202512,70,70
4,700636962155633,L400,Adalimumabe 40mg,202003,202512,70,70
5,700527448365900,L400,Adalimumabe 40mg,202002,202511,70,70
6,700257665328895,L400,Adalimumabe 40mg,202003,202512,70,70
7,700813983609726,L400,Adalimumabe 40mg,202002,202512,69,69
8,700448052079974,L400,Adalimumabe 40mg,202003,202512,69,69
9,700197371471188,L400,Adalimumabe 40mg,202004,202512,69,69


---

### 5. `CASE WHEN` (Colando a etiqueta de Faixa de Meses na capa — de 3 até 72 meses)

Como a nossa tabela `apac_psoriase` cobre **6 anos completos (72 meses: Janeiro/2020 a Dezembro/2025)**, o auditor organiza a escada do **`CASE WHEN`** cobrindo todas as faixas até os 72 meses:

1. **Até 3 meses (`<= 3`)** $\rightarrow$ `'1. Até 3 meses (1 a 3)'`
2. **Até 6 meses (`<= 6`)** $\rightarrow$ `'2. De 4 a 6 meses'`
3. **Até 12 meses (`<= 12` — 1 ano)** $\rightarrow$ `'3. De 7 a 12 meses (1 ano)'`
4. **Até 18 meses (`<= 18` — 1 ano e meio)** $\rightarrow$ `'4. De 13 a 18 meses (1,5 ano)'`
5. **Até 24 meses (`<= 24` — 2 anos)** $\rightarrow$ `'5. De 19 a 24 meses (2 anos)'`
6. **Até 36 meses (`<= 36` — 3 anos)** $\rightarrow$ `'6. De 25 a 36 meses (3 anos)'`
7. **Até 48 meses (`<= 48` — 4 anos)** $\rightarrow$ `'7. De 37 a 48 meses (4 anos)'`
8. **Até 60 meses (`<= 60` — 5 anos)** $\rightarrow$ `'8. De 49 a 60 meses (5 anos)'`
9. **Mais de 60 meses (`61 a 72` — até 6 anos)** $\rightarrow$ `'9. De 61 a 72 meses (6 anos)'`

In [42]:
%%sql
-- 5. Classificando cada paciente na sua faixa de meses utilizados (de 3 até 72 meses)
SELECT
    AP_CNSPCN AS paciente,
    MAX(AP_CIDPRI) AS cid10,
    'Adalimumabe 40mg' AS medicamento,
    MIN(AP_CMP) AS data_inicio,
    MAX(AP_CMP) AS data_final,
    COUNT(DISTINCT AP_CMP) AS total_meses_utilizados,
    COUNT(DISTINCT AP_AUTORIZ) AS qtd_apacs_distintas,
    CASE
        WHEN COUNT(DISTINCT AP_CMP) <= 3  THEN '1. Até 3 meses (1 a 3)'
        WHEN COUNT(DISTINCT AP_CMP) <= 6  THEN '2. De 4 a 6 meses'
        WHEN COUNT(DISTINCT AP_CMP) <= 12 THEN '3. De 7 a 12 meses (1 ano)'
        WHEN COUNT(DISTINCT AP_CMP) <= 18 THEN '4. De 13 a 18 meses (1,5 ano)'
        WHEN COUNT(DISTINCT AP_CMP) <= 24 THEN '5. De 19 a 24 meses (2 anos)'
        WHEN COUNT(DISTINCT AP_CMP) <= 36 THEN '6. De 25 a 36 meses (3 anos)'
        WHEN COUNT(DISTINCT AP_CMP) <= 48 THEN '7. De 37 a 48 meses (4 anos)'
        WHEN COUNT(DISTINCT AP_CMP) <= 60 THEN '8. De 49 a 60 meses (5 anos)'
        ELSE '9. De 61 a 72 meses (6 anos)'
    END AS faixa_meses
FROM apac_psoriase
WHERE AP_PRIPAL IN ('0604380011', '0604380062', '0604380127', '0604380135')
GROUP BY AP_CNSPCN
ORDER BY total_meses_utilizados DESC
LIMIT 20;

---

### 6. Contando quantos pacientes caíram em cada uma das 9 Faixas (`GROUP BY faixa_meses`)

Agora o auditor separa as pastas nas **9 caixas de faixa de tempo** (de `Até 3 meses` até `De 61 a 72 meses`) e conta quantos pacientes de Adalimumabe existem em cada faixa ao longo dos 6 anos (2020 a 2025).

In [46]:
%%sql
-- 6. Total de pacientes de Adalimumabe em cada uma das 9 faixas de meses (2020 a 2025)
SELECT
    faixa_meses,
    COUNT(*) AS total_pacientes
FROM (
    SELECT
        AP_CNSPCN AS paciente,
        COUNT(DISTINCT AP_CMP) AS total_meses_utilizados,
        CASE
            WHEN COUNT(DISTINCT AP_CMP) <= 3  THEN '1. Até 3 meses (1 a 3)'
            WHEN COUNT(DISTINCT AP_CMP) <= 6  THEN '2. De 4 a 6 meses'
            WHEN COUNT(DISTINCT AP_CMP) <= 12 THEN '3. De 7 a 12 meses (1 ano)'
            WHEN COUNT(DISTINCT AP_CMP) <= 18 THEN '4. De 13 a 18 meses (1,5 ano)'
            WHEN COUNT(DISTINCT AP_CMP) <= 24 THEN '5. De 19 a 24 meses (2 anos)'
            WHEN COUNT(DISTINCT AP_CMP) <= 36 THEN '6. De 25 a 36 meses (3 anos)'
            WHEN COUNT(DISTINCT AP_CMP) <= 48 THEN '7. De 37 a 48 meses (4 anos)'
            WHEN COUNT(DISTINCT AP_CMP) <= 60 THEN '8. De 49 a 60 meses (5 anos)'
            ELSE '9. De 61 a 72 meses (6 anos)'
        END AS faixa_meses
    FROM apac_psoriase
    WHERE AP_PRIPAL IN ('0604380011', '0604380062', '0604380127', '0604380135')
    GROUP BY AP_CNSPCN
) AS resumo_por_paciente
GROUP BY faixa_meses
ORDER BY faixa_meses;

,faixa_meses,total_pacientes
0,1. Até 3 meses (1 a 3),189
1,2. De 4 a 6 meses,248
2,3. De 7 a 12 meses (1 ano),314
3,"4. De 13 a 18 meses (1,5 ano)",218
4,5. De 19 a 24 meses (2 anos),134
5,6. De 25 a 36 meses (3 anos),173
6,7. De 37 a 48 meses (4 anos),98
7,8. De 49 a 60 meses (5 anos),72
8,9. De 61 a 72 meses (6 anos),34


---

### 7. Abrindo a pasta de um paciente específico (`WHERE AP_CNSPCN = '700503585571647'`)

Na Etapa 5, vimos na capa que o paciente **`700503585571647`** teve exatamente **5 APACs** de Adalimumabe.
Agora o auditor abre a pasta desse paciente (`WHERE AP_CNSPCN = '700503585571647'`) para ver na mesa todas as fichas individuais dele em ordem cronológica (`ORDER BY AP_CMP`) — inclusive para checar se depois do Adalimumabe ele trocou para outro medicamento de Psoríase ou se parou totalmente.

In [44]:
%%sql
-- 7. Abre a pasta do paciente '700503585571647' para ver linha por linha o histórico dele na Psoríase
SELECT
    AP_CMP AS ano_mes,
    AP_CNSPCN AS cns_paciente,
    AP_CIDPRI AS cid_psoriase,
    AP_PRIPAL AS codigo_medicamento,
    AP_AUTORIZ AS numero_ficha_apac,
    AP_MOTSAI AS motivo_saida
FROM apac_psoriase
WHERE AP_CNSPCN = '700503585571647'
ORDER BY ano_mes;

,ano_mes,cns_paciente,cid_psoriase,codigo_medicamento,numero_ficha_apac,motivo_saida
0,202001,700503585571647,L400,0604600020,4219203834769,21
1,202002,700503585571647,L400,0604600020,4219203834769,21
2,202003,700503585571647,L400,0604600020,4220201032201,21
3,202004,700503585571647,L400,0604600020,4220201032201,28
4,202005,700503585571647,L400,0604600020,4220201032201,21
5,202006,700503585571647,L400,0604600020,4220202169117,21
6,202007,700503585571647,L400,0604600020,4220202169117,28
7,202008,700503585571647,L400,0604600020,4220202169117,28
8,202009,700503585571647,L400,0604600020,4220202978772,28
9,202010,700503585571647,L400,0604380062,4220202978772,28
